# LeetCode 3298 — Count Substrings That Can Be Rearranged to Contain a String II bằng Python

- Link: https://leetcode.com/problems/count-substrings-that-can-be-rearranged-to-contain-a-string-ii/
- Độ khó: **Hard**
- Chủ đề: **Cửa sổ trượt** với bộ đếm **"còn thiếu"** (missing), tính chất đơn điệu ⇒ hai con trỏ `O(n)`; nhận ra "xếp lại để có `word2` làm tiền tố" ⇔ **đa tập chứa đa tập của `word2`**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def validSubstringCount(self, word1: str, word2: str) -> int:
```

## 1. Mục tiêu học

1. Dịch điều kiện: `x` xếp lại được để có `word2` làm tiền tố ⇔ với mọi chữ `c`, `cnt_x[c] ≥ cnt_{word2}[c]` (đặt `word2` lên đầu, phần dư phía sau).
2. Tính đơn điệu: nếu `word1[l..r]` hợp lệ thì mọi đoạn **chứa** nó cũng hợp lệ ⇒ với mỗi `r`, các `l` hợp lệ là tiền tố `0..L(r) − 1`.
3. Duy trì cửa sổ bằng biến `missing` = tổng lượng còn thiếu, cập nhật `O(1)` mỗi bước (không so 26 chữ mỗi lần).
4. Bộ nhớ hạn chế: chỉ dùng mảng 26 phần tử, không tạo xâu con.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho `word1`, `word2`. Xâu `x` **hợp lệ** nếu có thể sắp xếp lại `x` để `word2` là **tiền tố**. Đếm số xâu con hợp lệ (theo vị trí) của `word1`. Đề yêu cầu thời gian **tuyến tính** (giới hạn bộ nhớ nhỏ).

**Ví dụ 1:** `"bcca"`, `"abc"` → `1`. **Ví dụ 2:** `"abcabc"`, `"abc"` → `10`. **Ví dụ 3:** `"abcabc"`, `"aaabc"` → `0`.

**Ràng buộc:** `1 ≤ len(word1) ≤ 10^6`; `1 ≤ len(word2) ≤ 10^4`; chữ thường.

In [1]:
from typing import List
from collections import Counter

examples = [("bcca", "abc", 1), ("abcabc", "abc", 10), ("abcabc", "aaabc", 0)]
for w1, w2, exp in examples:
    print(repr(w1), repr(w2), '-> kỳ vọng', exp)

'bcca' 'abc' -> kỳ vọng 1
'abcabc' 'abc' -> kỳ vọng 10
'abcabc' 'aaabc' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. "Xếp lại có tiền tố" ⇔ chứa đa tập

Nếu `x` chứa đủ các chữ của `word2` (tính cả số lần), đặt chúng theo đúng thứ tự `word2` lên đầu, phần còn lại tùy ý ⇒ hợp lệ. Ngược lại thiếu chữ nào thì không thể.

In [2]:
def contains_multiset(x, y):
    cx, cy = Counter(x), Counter(y)
    return all(cx[c] >= v for c, v in cy.items())

print(contains_multiset("bcca", "abc"), contains_multiset("bca", "abcc"), contains_multiset("aab", "ab"))

True False True


### 3.2. Biến `missing`

`need[c]` = số chữ `c` còn **thiếu** trong cửa sổ (có thể âm = dư). `missing = Σ max(need[c], 0)`. Thêm chữ `c`: nếu `need[c] > 0` thì `missing −= 1`; `need[c] −= 1`. Bỏ chữ `c`: `need[c] += 1`; nếu sau đó `need[c] > 0` thì `missing += 1`. Cửa sổ hợp lệ ⇔ `missing = 0`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 `"abcabc"`, `"abc"`: với mỗi đầu trái `l`, đầu phải nhỏ nhất `r(l)` để đoạn chứa `a, b, c`:

| `l` | `r(l)` | số đầu phải hợp lệ `n − r(l)` |
|---|---|---|
| 0 | 2 | 4 |
| 1 | 3 | 3 |
| 2 | 4 | 2 |
| 3 | 5 | 1 |

Tổng `10` ✓. `r(l)` không giảm khi `l` tăng ⇒ hai con trỏ.

## 5. Cách 1 — Brute Force: mọi xâu con, đếm tăng dần theo đầu phải

In [3]:
def brute(w1, w2):
    n = len(w1)
    need = Counter(w2)
    total = 0
    for l in range(n):
        cnt = Counter()
        for r in range(l, n):
            cnt[w1[r]] += 1
            if all(cnt[c] >= v for c, v in need.items()):
                total += n - r          # mọi r' >= r cũng hợp lệ
                break
    return total

for w1, w2, exp in examples:
    print(brute(w1, w2), '| kỳ vọng', exp)

1 | kỳ vọng 1
10 | kỳ vọng 10
0 | kỳ vọng 0


**Độ phức tạp:** `O(n² · 26)`.

## 6. Ý tưởng thuật toán / bước đột phá

Cách "đếm theo đầu phải" (giống LC 3329):

```
need = cnt(word2); missing = len(word2); l = 0; ans = 0
for r, c in enumerate(word1):
    thêm c (cập nhật need, missing)
    while missing == 0:            # cửa sổ [l, r] hợp lệ -> thu hẹp
        bỏ word1[l]; l += 1
    ans += l                       # các đầu trái 0..l-1 cho đoạn hợp lệ kết thúc tại r
```

Sau vòng `while`, `[l, r]` là đoạn **không** hợp lệ ngắn nhất về bên trái, nên đúng `l` đầu trái cho đoạn hợp lệ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tìm kiếm nhị phân đầu phải với tổng tiền tố theo chữ

`P[c][i]` = số chữ `c` trong `word1[:i]`. Với mỗi `l`, nhị phân `r` nhỏ nhất thỏa mọi chữ. `O(n · 26 · log n)` thời gian và `O(26 n)` bộ nhớ — vi phạm giới hạn bộ nhớ của đề nhưng hữu ích để đối chiếu.

In [4]:
def binary_search(w1, w2):
    n = len(w1)
    need = Counter(w2)
    chars = list(need)
    P = {c: [0] * (n + 1) for c in chars}
    for i, ch in enumerate(w1):
        for c in chars:
            P[c][i + 1] = P[c][i] + (ch == c)
    total = 0
    for l in range(n):
        lo, hi = l, n                    # tìm r+1 nhỏ nhất trong [l+1, n]
        if any(P[c][n] - P[c][l] < need[c] for c in chars):
            continue
        lo = l + 1
        while lo < hi:
            mid = (lo + hi) // 2
            if all(P[c][mid] - P[c][l] >= need[c] for c in chars):
                hi = mid
            else:
                lo = mid + 1
        total += n - lo + 1
    return total

for w1, w2, exp in examples:
    print(binary_search(w1, w2), '| kỳ vọng', exp)

1 | kỳ vọng 1
10 | kỳ vọng 10
0 | kỳ vọng 0


### 7.2. Cách 3 — Hai con trỏ theo đầu trái (cộng `n − r`)

In [5]:
def two_pointers_left(w1, w2):
    n = len(w1)
    need = [0] * 26
    for c in w2:
        need[ord(c) - 97] += 1
    missing = len(w2)
    r = 0
    total = 0
    for l in range(n):
        while r < n and missing > 0:
            c = ord(w1[r]) - 97
            if need[c] > 0:
                missing -= 1
            need[c] -= 1
            r += 1
        if missing == 0:
            total += n - r + 1           # đoạn [l, r-1] là đoạn hợp lệ ngắn nhất bắt đầu tại l
        c = ord(w1[l]) - 97              # bỏ w1[l]
        need[c] += 1
        if need[c] > 0:
            missing += 1
    return total

for w1, w2, exp in examples:
    print(two_pointers_left(w1, w2), '| kỳ vọng', exp)

1 | kỳ vọng 1
10 | kỳ vọng 10
0 | kỳ vọng 0


### 7.3. Cách 4 — Hai con trỏ theo đầu phải (cộng `l`) — lời giải chính

In [6]:
def two_pointers_right(w1, w2):
    need = [0] * 26
    for c in w2:
        need[ord(c) - 97] += 1
    missing = len(w2)
    ans = l = 0
    for ch in w1:
        c = ord(ch) - 97
        if need[c] > 0:
            missing -= 1
        need[c] -= 1
        while missing == 0:
            d = ord(w1[l]) - 97
            need[d] += 1
            if need[d] > 0:
                missing += 1
            l += 1
        ans += l
    return ans

for w1, w2, exp in examples:
    print(two_pointers_right(w1, w2), '| kỳ vọng', exp)

1 | kỳ vọng 1
10 | kỳ vọng 10
0 | kỳ vọng 0


## 8. TRACE từng bước

Trace cách 4 trên `"abcabc"`, `"abc"`.

In [7]:
def trace(w1, w2):
    need = Counter(w2); missing = len(w2); ans = l = 0
    for r, ch in enumerate(w1):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        removed = ''
        while missing == 0:
            d = w1[l]; need[d] += 1
            if need[d] > 0:
                missing += 1
            removed += d; l += 1
        ans += l
        print(f"r = {r} ({ch}): bỏ {removed or '-':4s} l = {l}, missing = {missing} -> ans = {ans}")
    print('đáp án:', ans, '| vét cạn:', brute(w1, w2))

trace("abcabc", "abc")

r = 0 (a): bỏ -    l = 0, missing = 2 -> ans = 0
r = 1 (b): bỏ -    l = 0, missing = 1 -> ans = 0
r = 2 (c): bỏ a    l = 1, missing = 1 -> ans = 1
r = 3 (a): bỏ b    l = 2, missing = 1 -> ans = 3
r = 4 (b): bỏ c    l = 3, missing = 1 -> ans = 6
r = 5 (c): bỏ a    l = 4, missing = 1 -> ans = 10
đáp án: 10 | vét cạn: 10


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def validSubstringCount(self, word1: str, word2: str) -> int:
        need = [0] * 26
        for c in word2.encode():
            need[c - 97] += 1
        missing = len(word2)
        b = word1.encode()
        ans = l = 0
        for x in b:
            c = x - 97
            if need[c] > 0:
                missing -= 1
            need[c] -= 1
            while missing == 0:
                d = b[l] - 97
                need[d] += 1
                if need[d] > 0:
                    missing += 1
                l += 1
            ans += l
        return ans

for w1, w2, exp in examples:
    got = Solution().validSubstringCount(w1, w2)
    print(got, 'OK' if got == exp else 'SAI')

1 OK
10 OK
0 OK


`encode()` trả về `bytes` — duyệt nhanh và không tạo xâu con nào (bộ nhớ `O(1)` ngoài đầu vào).

## 10. Vì sao thuật toán đúng

**Bất biến** (sau xử lý `r`): `need[c] = cnt_{word2}[c] − cnt_{[l, r]}[c]`, `missing = Σ max(need[c], 0)`, cửa sổ `[l, r]` **không** hợp lệ, và `[l − 1, r]` hợp lệ (nếu `l > 0`).

- Cập nhật `missing`: thêm chữ `c` giảm thiếu hụt đúng 1 nếu `c` còn thiếu; bỏ chữ `c` tăng thiếu hụt nếu sau khi bỏ, `c` bị thiếu.
- Vòng `while` co `l` cho tới khi cửa sổ vừa mất tính hợp lệ.
- Do đơn điệu (mục 1.2), `[l', r]` hợp lệ ⇔ `l' < l` ⇒ cộng `l`. `l` không cần lùi vì `L(r)` không giảm theo `r`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn + break | `O(n² · 26)` | `O(26)` |
| 2. Nhị phân + tiền tố | `O(26 n log n)` | `O(26 n)` |
| 3. Hai con trỏ (đầu trái) | `O(n)` | `O(26)` |
| 4. Hai con trỏ (đầu phải) | `O(n)` | `O(26)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'binary_search': binary_search, 'two_left': two_pointers_left, 'two_right': two_pointers_right,
         'submit': lambda a, b: Solution().validSubstringCount(a, b)}
edge = [("a", "a", 1), ("a", "b", 0), ("aaa", "a", 6), ("aaa", "aa", 3), ("ab", "abc", 0), ("ba", "ab", 1), ("zzaz", "az", 5)]
for w1, w2, exp in examples + edge:
    assert brute(w1, w2) == exp, (w1, w2, brute(w1, w2))
    for name, f in impls.items():
        assert f(w1, w2) == exp, (name, w1, w2)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3298)
cnt = 0
for _ in range(800):
    w1 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 14)))
    w2 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 4)))
    exp = brute(w1, w2)
    for name, f in impls.items():
        assert f(w1, w2) == exp, (name, w1, w2)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 800 bộ test.


### 12.3. Ca lớn `len(word1) = 10^6`

In [11]:
import time
N = 10 ** 6
big = {
    'ngẫu nhiên, word2 = "abc"': (''.join(rng.choice('abcdefghij') for _ in range(N)), "abc"),
    'toàn a, word2 = "a"': ('a' * N, 'a'),
    'word2 dài 1e4': (''.join(rng.choice('ab') for _ in range(N)), ''.join(rng.choice('ab') for _ in range(10 ** 4))),
}
for name, (w1, w2) in big.items():
    t = time.time(); a = Solution().validSubstringCount(w1, w2); t1 = time.time() - t
    t = time.time(); b = two_pointers_left(w1, w2); t2 = time.time() - t
    assert a == b
    print(f"{name:28s}: {a:14d} | nộp {t1:.2f}s | cách 3 {t2:.2f}s")
assert Solution().validSubstringCount('a' * N, 'a') == N * (N + 1) // 2
print('toàn a: đáp án = n(n+1)/2 — đúng.')

ngẫu nhiên, word2 = "abc"   :   499983250551 | nộp 0.17s | cách 3 0.28s


toàn a, word2 = "a"         :   500000500000 | nộp 0.21s | cách 3 0.31s


word2 dài 1e4               :   489968570474 | nộp 0.21s | cách 3 0.32s
toàn a: đáp án = n(n+1)/2 — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hiểu nhầm là `word2` phải xuất hiện **liên tiếp** (không cho sắp lại)

In [12]:
def wrong_literal(w1, w2):
    n = len(w1)
    return sum(1 for l in range(n) for r in range(l, n) if w2 in w1[l:r + 1])   # SAI

print('sai :', wrong_literal("bcca", "abc"), '| đúng:', Solution().validSubstringCount("bcca", "abc"))

sai : 0 | đúng: 1


**Vì sao sai:** được phép **sắp xếp lại** `x` — chỉ cần đủ số lượng chữ.

### Lỗi 2 — Giảm `missing` cho **mọi** chữ (kể cả chữ đã dư)

In [13]:
def wrong_missing(w1, w2):
    need = Counter(w2); missing = len(w2); ans = l = 0
    for ch in w1:
        missing -= 1                                         # SAI: chỉ giảm khi chữ còn thiếu
        need[ch] -= 1
        while missing <= 0 and l < len(w1):
            d = w1[l]; need[d] += 1; missing += 1; l += 1
        ans += l
    return ans

print('sai :', wrong_missing("abcabc", "aaabc"), '| đúng:', Solution().validSubstringCount("abcabc", "aaabc"))

sai : 3 | đúng: 0


**Vì sao sai:** `missing` phải là thiếu hụt **thực**; chữ không có trong `word2` hoặc đã đủ không làm giảm thiếu hụt.

### Lỗi 3 — Cộng độ dài cửa sổ `r − l + 1`

In [14]:
def wrong_window(w1, w2):
    need = Counter(w2); missing = len(w2); ans = l = 0
    for r, ch in enumerate(w1):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:
            d = w1[l]; need[d] += 1
            if need[d] > 0:
                missing += 1
            l += 1
        ans += r - l + 1                                     # SAI: đếm các đoạn KHÔNG hợp lệ
    return ans

print('sai :', wrong_window("abcabc", "abc"), '| đúng:', Solution().validSubstringCount("abcabc", "abc"))

sai : 11 | đúng: 10


**Vì sao sai:** sau vòng `while`, cửa sổ `[l, r]` **không** hợp lệ; các đầu trái hợp lệ là `0..l−1`.

### Lỗi 4 — Cách 3: cộng `n − r` thay vì `n − r + 1` (con trỏ `r` là "một sau cuối")

In [15]:
def wrong_off_by_one(w1, w2):
    n = len(w1); need = Counter(w2); missing = len(w2); r = 0; total = 0
    for l in range(n):
        while r < n and missing > 0:
            ch = w1[r]
            if need[ch] > 0:
                missing -= 1
            need[ch] -= 1; r += 1
        if missing == 0:
            total += n - r                                   # SAI: đoạn [l, r-1] cũng hợp lệ
        need[w1[l]] += 1
        if need[w1[l]] > 0:
            missing += 1
    return total

print('sai :', wrong_off_by_one("bcca", "abc"), '| đúng:', Solution().validSubstringCount("bcca", "abc"))

sai : 0 | đúng: 1


**Vì sao sai:** `r` trỏ **sau** phần tử cuối của cửa sổ; các đầu phải hợp lệ là `r − 1, r, …, n − 1` — có `n − r + 1` giá trị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp lại được để chứa …" ⇒ chỉ còn là so sánh **đa tập**.
- "Đếm xâu con thỏa tính chất đơn điệu khi mở rộng" ⇒ hai con trỏ (cùng họ LC 76 Minimum Window Substring, LC 3329).
- Biến `missing` (hay `formed`) giúp kiểm tra điều kiện 26 chữ trong `O(1)`.

## 15. Biến thể của bài

1. **LC 3297 (bản I):** `len(word1) ≤ 10^5` — cùng lời giải.
2. **LC 76:** trả về đoạn **ngắn nhất** hợp lệ.
3. Đếm xâu con mà đa tập **bằng đúng** đa tập `word2` (hoán vị) ⇒ cửa sổ cố định độ dài `m`.
4. Đếm xâu con **thiếu tối đa `t`** chữ ⇒ điều kiện `missing ≤ t`.

In [16]:
# Biến thể 2, 3, 4
def min_window(w1, w2):
    need = Counter(w2); missing = len(w2); l = 0; best = None
    for r, ch in enumerate(w1):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:
            if best is None or r - l + 1 < best[1] - best[0]:
                best = (l, r + 1)
            d = w1[l]; need[d] += 1
            if need[d] > 0:
                missing += 1
            l += 1
    return w1[best[0]:best[1]] if best else ""

def count_anagrams(w1, w2):
    m = len(w2); target = Counter(w2)
    return sum(1 for i in range(len(w1) - m + 1) if Counter(w1[i:i + m]) == target)

def at_most_t_brute(w1, w2, t):
    n = len(w1); need = Counter(w2)
    return sum(1 for l in range(n) for r in range(l, n)
               if sum(max(0, v - Counter(w1[l:r + 1])[c]) for c, v in need.items()) <= t)

assert min_window("ADOBECODEBANC", "ABC") == "BANC"
rng = random.Random(15)
for _ in range(300):
    w1 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 10)))
    w2 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 3)))
    exp = [w1[l:r + 1] for l in range(len(w1)) for r in range(l, len(w1)) if contains_multiset(w1[l:r + 1], w2)]
    mw = min_window(w1, w2)
    assert (len(mw) == min(map(len, exp))) if exp else mw == ""
    assert count_anagrams(w1, w2) == sum(1 for i in range(len(w1) - len(w2) + 1) if sorted(w1[i:i + len(w2)]) == sorted(w2))
print('LC 76 ví dụ "BANC"; biến thể 2, 3 khớp vét cạn.')

LC 76 ví dụ "BANC"; biến thể 2, 3 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn + break | `O(26 n²)` | Oracle |
| 2 | Nhị phân + tiền tố | `O(26 n log n)` | Tốn bộ nhớ |
| 3 | Hai con trỏ (cộng `n − r + 1`) | `O(n)` | Dễ lệch một |
| 4 | Hai con trỏ (cộng `l`) | `O(n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"aaaa"`, `"aa"`.

**Bài 2 (tính tay):** `"abab"`, `"ab"`.

**Bài 3 (điền TODO):** đếm xâu con **không** hợp lệ (phần bù).

**Bài 4 (tự viết lại):** độ dài xâu con hợp lệ **ngắn nhất** (hoặc 0).

**Bài 5 (mini challenge):** biến thể 4 — xâu con thiếu tối đa `t` chữ.

In [17]:
def invalid_count_todo(w1, w2):
    # TODO
    raise NotImplementedError

def shortest_valid_todo(w1, w2):
    # TODO
    raise NotImplementedError

def at_most_t_todo(w1, w2, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đoạn có ≥ 2 chữ `a`. (2) Độ dài ≥ 2: `3 + 2 + 1`. (3) `6`.

**Bài 2.** (1) Đoạn chứa cả `a` và `b`. (2) Mọi đoạn độ dài ≥ 2. (3) `3 + 2 + 1 = 6`.

**Bài 3.** (1) `n(n+1)/2 − đáp án`. (2) Hoặc cộng `r − l + 1` (như lỗi 3!). (3) Kiểm tra cả hai.

**Bài 4.** (1) LC 76. (2) Ghi lại độ dài trong vòng `while`. (3) Trả 0 nếu không có.

**Bài 5.** (1) Đổi điều kiện co cửa sổ thành `missing ≤ t`. (2) Vẫn đơn điệu. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3, 4
assert Solution().validSubstringCount("aaaa", "aa") == 6
assert Solution().validSubstringCount("abab", "ab") == 6 == brute("abab", "ab")

def invalid_count(w1, w2):
    n = len(w1)
    return n * (n + 1) // 2 - Solution().validSubstringCount(w1, w2)

def shortest_valid(w1, w2):
    return len(min_window(w1, w2))

assert invalid_count("abcabc", "abc") == 11 == wrong_window("abcabc", "abc")
assert shortest_valid("bcca", "abc") == 4
print('Bài 1: 6 | Bài 2: 6 | Bài 3: 11 | Bài 4:', shortest_valid("bcca", "abc"))

Bài 1: 6 | Bài 2: 6 | Bài 3: 11 | Bài 4: 4


In [19]:
# Bài 5
def at_most_t(w1, w2, t):
    need = Counter(w2); missing = len(w2); ans = l = 0
    for ch in w1:
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing <= t:                  # [l, r] hợp lệ -> bỏ w1[l]
            d = w1[l]; need[d] += 1
            if need[d] > 0:
                missing += 1
            l += 1
        ans += l
    return ans

rng = random.Random(4)
for _ in range(300):
    w1 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 9)))
    w2 = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 4)))
    t = rng.randint(0, 2)
    if t >= len(w2):
        continue                       # mọi đoạn (kể cả rỗng) hợp lệ — loại ca suy biến
    assert at_most_t(w1, w2, t) == at_most_t_brute(w1, w2, t), (w1, w2, t)
print('Bài 5 khớp vét cạn.')

Bài 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
need = cnt(word2); missing = len(word2); ans = l = 0
for c in word1:
    if need[c] > 0: missing -= 1
    need[c] -= 1
    while missing == 0:
        need[word1[l]] += 1
        if need[word1[l]] > 0: missing += 1
        l += 1
    ans += l
```

**Checklist:**

- [ ] Điều kiện là **đa tập chứa đa tập** (không cần liên tiếp)?
- [ ] Chỉ giảm `missing` khi chữ còn thiếu (`need > 0`)?
- [ ] Cộng `l` (không phải độ dài cửa sổ)?
- [ ] Không tạo xâu con (giới hạn bộ nhớ)?
- [ ] `word2` dài hơn `word1` ⇒ 0?